# Exploring the data downloaded from USDA FoodData Central

See the download here: https://fdc.nal.usda.gov/download-datasets.html

Data available in `.data/`.

Data dictionary available in  `nutrify/data_exploration/data/FoodData_Central_foundation_food_csv_2021-04-28/Download & API Field Descriptions April 2021.pdf`





In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## Get Data

In [175]:
# Import databases
food = pd.read_csv("data/FoodData_Central_foundation_food_csv_2026-04-30/food.csv")
food_survey = pd.read_csv("data/FoodData_Central_survey_food_csv_2024-10-31/food.csv")
nutrient = pd.read_csv("data/FoodData_Central_Supporting_Data_csv_2022-10-28/nutrient.csv")
food_nutrient = pd.read_csv("data/FoodData_Central_foundation_food_csv_2026-04-30/food_nutrient.csv")
food_nutrient_survey = pd.read_csv("data/FoodData_Central_survey_food_csv_2024-10-31/food_nutrient.csv")

print(len(food), len(food_survey), len(nutrient), len(food_nutrient), len(food_nutrient_survey))

# Combine food and food_survey and drop columns that don't have a description 
food = pd.concat([food, food_survey], ignore_index=True)
food = food.dropna(subset=["description"])
food["description"] = food["description"].str.lower()
print(f"Combined food rows: {len(food)}")

# Combine food_nutrient and food_nutrient_survey
food_nutrient = pd.concat(
    [food_nutrient, food_nutrient_survey],
    ignore_index=True
)
food_nutrient["nutrient_name"] = food_nutrient["nutrient_id"].map(nutrient.set_index("id")["name"]).str.lower() 
print(f"Combined food nutrient rows: {len(food_nutrient)}")

C:\Users\pyaes\AppData\Local\Temp\ipykernel_18888\2043736291.py:5: DtypeWarning: Columns (0: footnote) have mixed types. Specify dtype option on import or set low_memory=False.
  food_nutrient = pd.read_csv("data/FoodData_Central_foundation_food_csv_2026-04-30/food_nutrient.csv")


87990 5432 474 170469 353015
Combined food rows: 93414
Combined food nutrient rows: 523484


In [176]:
nutrient[nutrient["name"].str.contains("Energy")]

,id,name,unit_name,nutrient_nbr,rank
0,2047,Energy (Atwater General Factors),KCAL,957.0,280.0
1,2048,Energy (Atwater Specific Factors),KCAL,958.0,290.0
9,1008,Energy,KCAL,208.0,300.0
63,1062,Energy,kJ,268.0,400.0


In [177]:
food_nutrient.columns

Index(['id', 'fdc_id', 'nutrient_id', 'amount', 'data_points', 'derivation_id',
       'min', 'max', 'median', 'footnote', 'min_year_acquired',
       'nutrient_name'],
      dtype='str')

In [178]:
len(food_nutrient)

523484

In [179]:
food.head()

,fdc_id,data_type,description,food_category_id,publication_date
0,319874,sample_food,"hummus, sabra classic",16.0,2019-04-01
1,319875,market_acquisition,"hummus, sabra classic",16.0,2019-04-01
2,319876,market_acquisition,"hummus, sabra classic",16.0,2019-04-01
3,319877,sub_sample_food,hummus,16.0,2019-04-01
4,319878,sub_sample_food,hummus,16.0,2019-04-01


In [180]:
food_nutrient.head()

,id,fdc_id,nutrient_id,amount,data_points,derivation_id,min,max,median,footnote,min_year_acquired,nutrient_name
0,2201847,319877,1051,56.30,1.0,1.0,NaN,NaN,NaN,NaN,NaN,water
1,2201845,319877,1002,1.28,1.0,1.0,NaN,NaN,NaN,NaN,NaN,nitrogen
2,2201846,319877,1004,19.00,1.0,1.0,NaN,NaN,NaN,NaN,NaN,total lipid (fat)
3,2201844,319877,1007,1.98,1.0,1.0,NaN,NaN,NaN,NaN,NaN,ash
4,2201852,319878,1091,188.00,1.0,1.0,NaN,NaN,NaN,NaN,NaN,"phosphorus, p"


In [181]:
# How many unique?
unique_descriptions = food["description"].unique()
len(unique_descriptions)

17130

Beautiful, this gives us ~11368 foods to work with as a goal to model. But surely they can be split into less categories?

In [182]:
unique_descriptions[:10]

<ArrowStringArray>
['hummus, sabra classic',                'hummus',         'hummus, other',
    'hummus - nfy12140o',    'hummus - nfy12140p',    'hummus - nfy12140q',
    'hummus - nfy12140r',    'hummus - nfy12140s',    'hummus - nfy12140f',
    'hummus - nfy12140g']
Length: 10, dtype: str

Where do these descriptions come from?

How can we reduce them down to like 10 unique foods and keep it simple...

In [183]:
unique_descriptions[-10:]

<ArrowStringArray>
[                'celery, cooked, as ingredient',
 'dark green vegetables as ingredient in omelet',
              'tomatoes as ingredient in omelet',
      'other vegetables as ingredient in omelet',
               'mirepoix, cooked, as ingredient',
             'vegetables as ingredient in curry',
             'vegetables as ingredient in soups',
             'vegetables as ingredient in stews',
             'sauce as ingredient in hamburgers',
          'industrial oil as ingredient in food']
Length: 10, dtype: str

In [184]:
# Find random indexes of food to explore
import random
random_number = random.randint(0, len(unique_descriptions)-10)
unique_descriptions[random_number:random_number+10]

<ArrowStringArray>
[                             'pastry, cookie type, fried',
                            'pastry, italian, with cheese',
                                            'pastry, puff',
 'pastry, puff, custard or cream filled, iced or not iced',
                                     'cheese pastry puffs',
                   'pastry, mainly flour and water, fried',
                                         'empanada, fruit',
                                   'breakfast pastry, nfs',
                           'danish pastry, plain or spice',
                               'danish pastry, with fruit']
Length: 10, dtype: str

### Food Categories

Let's dive into food categories. 

In [185]:
food.head()

,fdc_id,data_type,description,food_category_id,publication_date
0,319874,sample_food,"hummus, sabra classic",16.0,2019-04-01
1,319875,market_acquisition,"hummus, sabra classic",16.0,2019-04-01
2,319876,market_acquisition,"hummus, sabra classic",16.0,2019-04-01
3,319877,sub_sample_food,hummus,16.0,2019-04-01
4,319878,sub_sample_food,hummus,16.0,2019-04-01


In [186]:
unique_categories = food["food_category_id"].unique()
unique_categories

array([1.600e+01, 1.000e+00, 1.300e+01, 1.100e+01, 2.000e+00, 7.000e+00,
       1.200e+01, 6.000e+00, 9.000e+00, 1.800e+01, 4.000e+00, 5.000e+00,
       1.500e+01, 1.900e+01, 2.500e+01, 1.000e+01,       nan, 2.000e+01,
       1.400e+01, 1.700e+01, 9.602e+03, 1.004e+03, 1.002e+03, 1.006e+03,
       1.008e+03, 1.202e+03, 1.902e+03, 1.820e+03, 1.822e+03, 8.412e+03,
       5.802e+03, 9.007e+03, 9.010e+03, 1.206e+03, 1.204e+03, 1.208e+03,
       1.402e+03, 7.220e+03, 9.404e+03, 9.402e+03, 9.999e+03, 8.008e+03,
       8.006e+03, 5.804e+03, 5.502e+03, 1.602e+03, 1.604e+03, 3.602e+03,
       2.502e+03, 3.720e+03, 6.432e+03, 2.002e+03, 2.004e+03, 9.008e+03,
       2.602e+03, 2.604e+03, 2.006e+03, 3.002e+03, 8.002e+03, 2.008e+03,
       2.206e+03, 2.202e+03, 2.204e+03, 2.010e+03, 2.606e+03, 2.608e+03,
       2.402e+03, 2.404e+03, 3.404e+03, 3.004e+03, 8.410e+03, 3.006e+03,
       8.404e+03, 3.202e+03, 3.402e+03, 3.502e+03, 6.411e+03, 3.740e+03,
       3.742e+03, 3.702e+03, 3.704e+03, 3.730e+03, 

19 different food categories... I wonder what these are?

In [187]:
food["food_category_id"].unique()

array([1.600e+01, 1.000e+00, 1.300e+01, 1.100e+01, 2.000e+00, 7.000e+00,
       1.200e+01, 6.000e+00, 9.000e+00, 1.800e+01, 4.000e+00, 5.000e+00,
       1.500e+01, 1.900e+01, 2.500e+01, 1.000e+01,       nan, 2.000e+01,
       1.400e+01, 1.700e+01, 9.602e+03, 1.004e+03, 1.002e+03, 1.006e+03,
       1.008e+03, 1.202e+03, 1.902e+03, 1.820e+03, 1.822e+03, 8.412e+03,
       5.802e+03, 9.007e+03, 9.010e+03, 1.206e+03, 1.204e+03, 1.208e+03,
       1.402e+03, 7.220e+03, 9.404e+03, 9.402e+03, 9.999e+03, 8.008e+03,
       8.006e+03, 5.804e+03, 5.502e+03, 1.602e+03, 1.604e+03, 3.602e+03,
       2.502e+03, 3.720e+03, 6.432e+03, 2.002e+03, 2.004e+03, 9.008e+03,
       2.602e+03, 2.604e+03, 2.006e+03, 3.002e+03, 8.002e+03, 2.008e+03,
       2.206e+03, 2.202e+03, 2.204e+03, 2.010e+03, 2.606e+03, 2.608e+03,
       2.402e+03, 2.404e+03, 3.404e+03, 3.004e+03, 8.410e+03, 3.006e+03,
       8.404e+03, 3.202e+03, 3.402e+03, 3.502e+03, 6.411e+03, 3.740e+03,
       3.742e+03, 3.702e+03, 3.704e+03, 3.730e+03, 

In [188]:
# Get food categories
food_cats = pd.read_csv("data/FoodData_Central_Supporting_Data_csv_2021-04-28/food_category.csv")
food_cats["id"].unique()

array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
       18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28])

## 10 foods we want

To keep things simple, we will reduce the databases from FoodData Central to 10 different foods.

Why these foods?

Because we have images for those foods ready to go.

```python
# These aren't whole foods so we don't want them yet, let's get another list and get those
ten_foods = ["chicken_curry", 
"chicken_wings", 
"fried_rice", 
"grilled_salmon", 
"humburger", 
"ice_cream", 
"pizza",
"ramen", 
"steak", 
"sushi"]

# We want these... (they're whole foods) 
ten_whole_foods = ["chicken_wings",
    "apple",
    "banana",
    "beef", # steak, etc
    "carrots",
    "egg", # whole egg
    "strawberries",
    "blueberries",
    "mushrooms",
    "honey"
]
```

In [189]:
ten_whole_foods = ['apple',
 'banana',
 'beef', # steak etc
 'blueberries',
 'carrots',
 'chicken_wings',
 'egg', # whole egg
 'honey',
 'mushrooms',
 'strawberries']
ten_whole_foods

['apple',
 'banana',
 'beef',
 'blueberries',
 'carrots',
 'chicken_wings',
 'egg',
 'honey',
 'mushrooms',
 'strawberries']

In [190]:
food.head()

,fdc_id,data_type,description,food_category_id,publication_date
0,319874,sample_food,"hummus, sabra classic",16.0,2019-04-01
1,319875,market_acquisition,"hummus, sabra classic",16.0,2019-04-01
2,319876,market_acquisition,"hummus, sabra classic",16.0,2019-04-01
3,319877,sub_sample_food,hummus,16.0,2019-04-01
4,319878,sub_sample_food,hummus,16.0,2019-04-01


In [191]:
# Foundation food is the ground truth for a certain type of food, excludes some details about the food
# E.g. the data_type foundation_food for Chicken will the the original unique ID for chicken
foundation_food = food[(food["data_type"] == "foundation_food") | (food["data_type"] == "survey_fndds_food")]
len(foundation_food)

5901

In [192]:
foundation_food[foundation_food["description"].str.contains("blue")]

,fdc_id,data_type,description,food_category_id,publication_date
42186,2263889,foundation_food,"blueberries, raw",9.0,2022-04-28
43441,2346411,foundation_food,"blueberries, raw",9.0,2022-10-28
60660,2684446,foundation_food,"crustaceans, crab, blue swimming, lump, pasteu...",15.0,2024-04-18
88312,2705705,survey_fndds_food,"cheese, blue or roquefort",1602.0,2022-10-28
90605,2707998,survey_fndds_food,"pie, blueberry",5502.0,2022-10-28
91805,2709198,survey_fndds_food,"blueberries, dried",6016.0,2022-10-28
91882,2709275,survey_fndds_food,"blueberries, raw",6011.0,2022-10-28
91884,2709277,survey_fndds_food,"blueberries, frozen",6011.0,2022-10-28
91885,2709278,survey_fndds_food,blueberry pie filling,8806.0,2022-10-28
91930,2709323,survey_fndds_food,blueberry juice,7006.0,2022-10-28


In [193]:
foundation_foods = foundation_food["description"]
foundation_foods[20:40]

4153               peanut butter, smooth style, with salt
4329                             cheese, parmesan, grated
4491    cheese, pasteurized process, american, vitamin...
4580    grapefruit juice, white, canned or bottled, un...
4723                                 peaches, yellow, raw
4817    seeds, sunflower seed kernels, dry roasted, wi...
4951      sausage, italian, pork, mild, cooked, pan-fried
5164                  bread, white, commercially prepared
5285          sausage, turkey, breakfast links, mild, raw
5428                                        cheese, swiss
5489    kale, frozen, cooked, boiled, drained, without...
5751    carrots, frozen, unprepared (includes foods fo...
5991                            mustard, prepared, yellow
6198                                figs, dried, uncooked
6339                                kiwifruit, green, raw
6491                              melons, cantaloupe, raw
6650                                      nectarines, raw
6794    orange

In [194]:
# Found a list of the foundation foods we're going to start with!
foundation_foods_list = list(foundation_foods)
for food in foundation_foods_list:
    if "blue" in food:
        print(food)

blueberries, raw
blueberries, raw
crustaceans, crab, blue swimming, lump, pasteurized, refrigerated
cheese, blue or roquefort
pie, blueberry
blueberries, dried
blueberries, raw
blueberries, frozen
blueberry pie filling
blueberry juice
blue or roquefort cheese dressing
blue or roquefort cheese dressing, light
blue or roquefort cheese dressing, fat free
blueberry syrup


In [195]:
# food.loc[(food["description"].str.contains("chicken", case=False)) & (food["description"].str.contains("drumstick", case=False))][-10:]
# Find chicken in foundation food
for food in foundation_foods:
    if "chicken" in food.lower():
        print(food)

chicken, broilers or fryers, drumstick, meat only, cooked, braised
chicken, broiler or fryers, breast, skinless, boneless, meat only, cooked, braised
chicken, ground, with additives, raw
chicken, breast, boneless, skinless, raw
chicken, thigh, boneless, skinless, raw
chicken, drumstick, meat and skin, raw
chicken, thigh, meat and skin, raw
chicken, wing, meat and skin, raw
chicken, breast, meat and skin, raw
lunchmeat, chicken breast, sliced
mock chicken legs
chicken, ns as to part and cooking method, ns as to skin eaten
chicken, ns as to part and cooking method, skin eaten
chicken, ns as to part and cooking method, skin not eaten
chicken, ns as to part, baked, broiled, or roasted, ns as to skin eaten
chicken, ns as to part, baked, broiled, or roasted, skin eaten
chicken, ns as to part, baked, broiled, or roasted, skin not eaten
chicken, ns as to part, rotisserie, ns as to skin eaten
chicken, ns as to part, rotisserie, skin eaten
chicken, ns as to part, rotisserie, skin not eaten
chick

In [196]:
chicken_wing_id = int(foundation_food.loc[foundation_food["description"].str.contains("Chicken", case=False)].iloc[0]["fdc_id"])
chicken_wing_id

331897

In [197]:
apple_id = int(foundation_food.loc[foundation_food["description"].str.contains("Apple", case=False)].iloc[0]["fdc_id"])
apple_id

1105430

In [198]:
food_nutrient[food_nutrient["fdc_id"] == chicken_wing_id]

,id,fdc_id,nutrient_id,amount,data_points,derivation_id,min,max,median,footnote,min_year_acquired,nutrient_name
41714,2259068,331897,1303,0.003,5.0,1.0,0.002,0.004,0.003,NaN,2010.0,tfa 16:1 t
41715,2259065,331897,1280,0.008,5.0,1.0,0.008,0.009,0.008,NaN,2010.0,pufa 22:5 n-3 (dpa)
41716,2259076,331897,1404,0.045,5.0,1.0,0.035,0.059,0.042,NaN,2010.0,"pufa 18:3 n-3 c,c,c (ala)"
41717,2259059,331897,1261,0.002,5.0,1.0,0.001,0.003,0.002,NaN,2010.0,sfa 8:0
41718,2259106,331897,1109,0.170,1.0,1.0,NaN,NaN,0.170,NaN,2010.0,vitamin e (alpha-tocopherol)
...,...,...,...,...,...,...,...,...,...,...,...,...
41806,2259112,331897,1167,5.050,5.0,1.0,4.890,5.240,5.050,NaN,2010.0,niacin
41807,2259074,331897,1329,0.021,NaN,4.0,NaN,NaN,NaN,NaN,NaN,"fatty acids, total trans-monoenoic"
41808,2259138,331897,1330,0.008,NaN,4.0,NaN,NaN,NaN,NaN,NaN,"fatty acids, total trans-dienoic"
41809,13338545,331897,2047,149.000,NaN,1.0,NaN,NaN,NaN,NaN,NaN,energy (atwater general factors)


## Get protein, carb, fat IDs

See this document for info on foundation foods and their nutrients - https://fdc.nal.usda.gov/docs/Foundation_Foods_Documentation_Apr2021.pdf

* Carbohydrate, by difference = total carbohydrates


In [199]:
nutrient[(nutrient["name"].str.contains("protein", case=False)) | \
         (nutrient["name"].str.contains("carbohydrate", case=False)) | \
         (nutrient["name"].str.contains("fat", case=False)) | \
         (nutrient["name"].str.contains("energy", case=False))]

,id,name,unit_name,nutrient_nbr,rank
0,2047,Energy (Atwater General Factors),KCAL,957.0,280.0
1,2048,Energy (Atwater Specific Factors),KCAL,958.0,290.0
4,1003,Protein,G,203.0,600.0
5,1004,Total lipid (fat),G,204.0,800.0
6,1005,"Carbohydrate, by difference",G,205.0,1110.0
9,1008,Energy,KCAL,208.0,300.0
50,1049,"Solids, non-fat",G,253.0,999999.0
51,1050,"Carbohydrate, by summation",G,205.2,1120.0
54,1053,Adjusted Protein,G,257.0,700.0
63,1062,Energy,kJ,268.0,400.0


In [200]:
target_nutrients = nutrient[nutrient["name"].isin(["Protein", "Total lipid (fat)", "Carbohydrate, by difference", "Energy (Atwater Specific Factors)"])]

# target_nutrients = target_nutrients[
#     ~((nutrient["name"] == "Energy") & (nutrient["unit_name"] == "kJ"))
# ]
target_nutrients

,id,name,unit_name,nutrient_nbr,rank
1,2048,Energy (Atwater Specific Factors),KCAL,958.0,290.0
4,1003,Protein,G,203.0,600.0
5,1004,Total lipid (fat),G,204.0,800.0
6,1005,"Carbohydrate, by difference",G,205.0,1110.0


In [201]:
target_nutrient_dict = {1003: "protein",
    1004: "fat",
    1005: "carbohydrate",
    2048: "Energy (Atwater Specific Factors)"
}
target_nutrient_dict

{1003: 'protein',
 1004: 'fat',
 1005: 'carbohydrate',
 2048: 'Energy (Atwater Specific Factors)'}

## Get target food protein, fat, carbohydrates

We want to now index on the target foods and the target nutrients and retrieve their values for each food/nutrient.

E.g.

```python
{"food_1": {"protein": 100,
            "carbohydrate": 50,
            "fat": 20},
 "food_2": ...

...}
```

In [202]:
list(target_nutrient_dict.keys())

[1003, 1004, 1005, 2048]

In [203]:
food_nutrient

,id,fdc_id,nutrient_id,amount,data_points,derivation_id,min,max,median,footnote,min_year_acquired,nutrient_name
0,2201847,319877,1051,56.30,1.0,1.0,NaN,NaN,NaN,NaN,NaN,water
1,2201845,319877,1002,1.28,1.0,1.0,NaN,NaN,NaN,NaN,NaN,nitrogen
2,2201846,319877,1004,19.00,1.0,1.0,NaN,NaN,NaN,NaN,NaN,total lipid (fat)
3,2201844,319877,1007,1.98,1.0,1.0,NaN,NaN,NaN,NaN,NaN,ash
4,2201852,319878,1091,188.00,1.0,1.0,NaN,NaN,NaN,NaN,NaN,"phosphorus, p"
...,...,...,...,...,...,...,...,...,...,...,...,...
523479,34489112,2710814,208,892.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
523480,34489151,2710814,601,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
523481,34489135,2710814,337,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
523482,34489121,2710814,304,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [204]:
food_nutrient[(food_nutrient["nutrient_id"].isin(list(target_nutrient_dict.keys())))]

,id,fdc_id,nutrient_id,amount,data_points,derivation_id,min,max,median,footnote,min_year_acquired,nutrient_name
2,2201846,319877,1004,19.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,total lipid (fat)
16,2201859,319882,1004,18.7,1.0,1.0,NaN,NaN,NaN,NaN,NaN,total lipid (fat)
28,2201873,319892,1004,16.6,1.0,1.0,NaN,NaN,NaN,NaN,NaN,total lipid (fat)
43,2201886,319899,1004,19.1,1.0,1.0,NaN,NaN,NaN,NaN,NaN,total lipid (fat)
97,2201942,319908,1004,18.2,1.0,1.0,NaN,NaN,NaN,NaN,NaN,total lipid (fat)
...,...,...,...,...,...,...,...,...,...,...,...,...
169976,35088854,2768447,1003,17.9,1.0,1.0,NaN,NaN,NaN,NaN,NaN,protein
169977,35088855,2768448,1003,17.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,protein
169978,35088856,2768449,1003,17.1,1.0,1.0,NaN,NaN,NaN,NaN,NaN,protein
169979,35088857,2768450,1003,16.3,1.0,1.0,NaN,NaN,NaN,NaN,NaN,protein


In [205]:
unique_nutrients = food_nutrient[
    food_nutrient["nutrient_id"].isin(list(target_nutrient_dict.keys()))
]["nutrient_id"].unique()

print(unique_nutrients)

[1004 1003 1005 2048]


In [206]:
food_nutrient.dtypes

id                     int64
fdc_id                 int64
nutrient_id            int64
amount               float64
data_points          float64
derivation_id        float64
min                  float64
max                  float64
median               float64
footnote              object
min_year_acquired    float64
nutrient_name            str
dtype: object

In [207]:
# Find nutrition for chicken_wing_id (protein, fat, carb)
food_nutrient[(food_nutrient["fdc_id"] == chicken_wing_id) & (food_nutrient["nutrient_id"].isin(list(target_nutrient_dict.keys())))]

,id,fdc_id,nutrient_id,amount,data_points,derivation_id,min,max,median,footnote,min_year_acquired,nutrient_name
41750,2259098,331897,1004,5.95,6.0,1.0,5.54,6.33,5.93,NaN,2010.0,total lipid (fat)
41782,2259079,331897,1003,23.90,NaN,49.0,23.00,24.60,24.10,NaN,NaN,protein
41793,2259099,331897,1005,0.00,NaN,49.0,NaN,NaN,NaN,NaN,NaN,"carbohydrate, by difference"
41810,13338447,331897,2048,156.00,NaN,49.0,NaN,NaN,NaN,NaN,NaN,energy (atwater specific factors)


In [208]:
sorted(list(foundation_foods))

['abalone',
 'adobo, with noodles',
 'adobo, with rice',
 'agave liquid sweetener',
 'alaska pollock, raw',
 'alcoholic coffee drink',
 'alcoholic malt beverage',
 'alcoholic malt beverage, sweetened',
 'alfalfa sprouts, raw',
 'alfredo sauce',
 'alfredo sauce with added vegetables',
 'alfredo sauce with meat',
 'alfredo sauce with meat and added vegetables',
 'alfredo sauce with poultry',
 'alfredo sauce with poultry and added vegetables',
 'alfredo sauce with seafood',
 'alfredo sauce with seafood and added vegetables',
 'almond butter',
 'almond butter and jelly sandwich, on wheat bread',
 'almond butter and jelly sandwich, on white bread',
 'almond butter sandwich, on wheat bread',
 'almond butter sandwich, on white bread',
 'almond butter, creamy',
 'almond butter, lower sodium',
 'almond chicken',
 'almond milk, chocolate',
 'almond milk, nfs',
 'almond milk, sweetened',
 'almond milk, unsweetened',
 'almond milk, unsweetened, plain, refrigerated',
 'almond milk, unsweetened, pla

In [209]:
ten_whole_foods = ["chicken_wings",
    "apple",
    "banana",
    "beef", # steak, etc
    "carrots",
    "egg", # whole egg
    "strawberries",
    "blueberries",
    "mushrooms",
    "honey"
]

In [210]:
ten_whole_foods

['chicken_wings',
 'apple',
 'banana',
 'beef',
 'carrots',
 'egg',
 'strawberries',
 'blueberries',
 'mushrooms',
 'honey']

## Get ten whole foods `food_id`

Everything except blueberries and honey are available in `foundation_food`. 

For blueberries and honey, we'll have to dig into the survery data: `data_exploration/data/FoodData_Central_survey_food_csv_2020-10-30`

In [211]:
# Get all food ids from foundation_food (honey and blueberries in another dataset)
target_whole_foods = ['apple', # removed chicken wings... can come back later...
 'banana',
 'beef',
 'blueberries',
 'carrots',
 'chicken',
 'egg',
 'honey',
 'strawberries',
 'mushrooms']

In [212]:
# str.contains can search on regex - https://stackoverflow.com/a/17973255/7900723
pattern = "|".join([f"(?i){food}" for food in target_whole_foods])
pattern

'(?i)apple|(?i)banana|(?i)beef|(?i)blueberries|(?i)carrots|(?i)chicken|(?i)egg|(?i)honey|(?i)strawberries|(?i)mushrooms'

In [213]:
foundation_food[foundation_food["description"].str.contains(pattern, case=False)].sort_values(by=["description"])

,fdc_id,data_type,description,food_category_id,publication_date
89404,2706797,survey_fndds_food,almond chicken,3404.0,2022-10-28
90098,2707491,survey_fndds_food,"almonds, honey roasted",2804.0,2022-10-28
91926,2709319,survey_fndds_food,apple cider,7004.0,2022-10-28
93198,2710591,survey_fndds_food,"apple juice beverage, 40-50% juice, light",7204.0,2022-10-28
91927,2709320,survey_fndds_food,"apple juice, 100%",7004.0,2022-10-28
...,...,...,...,...,...
89271,2706664,survey_fndds_food,"venison or deer, noodles, and vegetables inclu...",3002.0,2022-10-28
89270,2706663,survey_fndds_food,"venison or deer, potatoes, and vegetables excl...",3002.0,2022-10-28
89269,2706662,survey_fndds_food,"venison or deer, potatoes, and vegetables incl...",3002.0,2022-10-28
90138,2707531,survey_fndds_food,"walnuts, excluding honey roasted",2804.0,2022-10-28


In [214]:
foundation_food[foundation_food["description"].str.contains("honey")]

,fdc_id,data_type,description,food_category_id,publication_date
20191,1105547,foundation_food,"apples, honeycrisp, with skin, raw",9.0,2020-10-30
20547,1750343,foundation_food,"apples, honeycrisp, with skin, raw",9.0,2020-10-30
63976,2710816,foundation_food,"melons, honeydew, raw",9.0,2024-10-31
90098,2707491,survey_fndds_food,"almonds, honey roasted",2804.0,2022-10-28
90105,2707498,survey_fndds_food,"cashews, honey roasted",2804.0,2022-10-28
90118,2707511,survey_fndds_food,"mixed nuts, honey roasted",2804.0,2022-10-28
90127,2707520,survey_fndds_food,"peanuts, honey roasted",2804.0,2022-10-28
90132,2707525,survey_fndds_food,"pecans, honey roasted",2804.0,2022-10-28
90138,2707531,survey_fndds_food,"walnuts, excluding honey roasted",2804.0,2022-10-28
90139,2707532,survey_fndds_food,"walnuts, honey roasted",2804.0,2022-10-28


In [215]:
# Found this earlier
chicken_wing_id

331897

In [216]:
# Map foods to food_id (these have been filtered from larger quantities to smaller quantities)
# For example, if there were 5 kinds of apple, only one was chosen
whole_foods_id_map = {1750339: "apple", # red delicious
    1105314: "banana", # Bananas, ripe and slightly ripe, raw
    1102702: "blueberries", # blueberries, raw	
    746763: "beef", # t-bone steak 
    746764: "carrots", # frozen unprepared
    331897: "chicken_wings", # Chicken, broilers or fryers, drumstick, meat o...	
    329490: "egg", # Egg, whole, dried	
    1103956: "honey", # Honey
    1750347: "mushrooms", # Mushrooms, white button
    747448: "strawberries" # strawberries, raw
}

In [217]:
list(whole_foods_id_map.keys())

[1750339,
 1105314,
 1102702,
 746763,
 746764,
 331897,
 329490,
 1103956,
 1750347,
 747448]

In [218]:
# Find nutrition for eight whole foods
target_whole_foods_df = food_nutrient[(food_nutrient["fdc_id"].isin(list(whole_foods_id_map.keys()))) & \
    (food_nutrient["nutrient_id"].isin(list(target_nutrient_dict.keys())))][["fdc_id", "nutrient_id", "amount"]]
target_whole_foods_df

,fdc_id,nutrient_id,amount
34317,329490,1004,39.800000
34318,329490,1005,1.870000
34322,329490,1003,48.100000
34339,329490,2048,576.000000
41750,331897,1004,5.950000
41782,331897,1003,23.900000
41793,331897,1005,0.000000
41810,331897,2048,156.000000
71149,746763,1003,27.300000
71176,746763,1005,0.000000


In [219]:
# Pivot the table to how we want it
target_whole_foods_df = target_whole_foods_df.pivot_table("amount", "fdc_id", "nutrient_id")
target_whole_foods_df

nutrient_id,1003,1004,1005,2048
fdc_id,,,,
329490,48.100000,39.8000,1.870000,576.000000
331897,23.900000,5.9500,0.000000,156.000000
746763,27.300000,11.4000,0.000000,219.000000
746764,0.810000,0.4700,7.920000,37.000000
747448,0.640000,0.2200,7.630000,31.000000
1105314,0.740000,0.2900,23.000000,88.000000
1750339,0.187500,0.2125,14.781700,55.622745
1750347,2.890625,0.3708,4.079375,24.873258


In [220]:
len(whole_foods_id_map)

10

In [221]:
target_whole_foods_df = target_whole_foods_df.reset_index(drop=False).rename_axis(None, axis=1)
target_whole_foods_df

,fdc_id,1003,1004,1005,2048
0,329490,48.100000,39.8000,1.870000,576.000000
1,331897,23.900000,5.9500,0.000000,156.000000
2,746763,27.300000,11.4000,0.000000,219.000000
3,746764,0.810000,0.4700,7.920000,37.000000
4,747448,0.640000,0.2200,7.630000,31.000000
5,1105314,0.740000,0.2900,23.000000,88.000000
6,1750339,0.187500,0.2125,14.781700,55.622745
7,1750347,2.890625,0.3708,4.079375,24.873258


In [222]:
target_nutrient_dict

{1003: 'protein',
 1004: 'fat',
 1005: 'carbohydrate',
 2048: 'Energy (Atwater Specific Factors)'}

In [223]:
# Rename columns
target_whole_foods_df.rename(columns=target_nutrient_dict, inplace=True)
target_whole_foods_df

,fdc_id,protein,fat,carbohydrate,Energy (Atwater Specific Factors)
0,329490,48.100000,39.8000,1.870000,576.000000
1,331897,23.900000,5.9500,0.000000,156.000000
2,746763,27.300000,11.4000,0.000000,219.000000
3,746764,0.810000,0.4700,7.920000,37.000000
4,747448,0.640000,0.2200,7.630000,31.000000
5,1105314,0.740000,0.2900,23.000000,88.000000
6,1750339,0.187500,0.2125,14.781700,55.622745
7,1750347,2.890625,0.3708,4.079375,24.873258


In [225]:
whole_foods_id_map

{1750339: 'apple',
 1105314: 'banana',
 1102702: 'blueberries',
 746763: 'beef',
 746764: 'carrots',
 331897: 'chicken_wings',
 329490: 'egg',
 1103956: 'honey',
 1750347: 'mushrooms',
 747448: 'strawberries'}

In [227]:
# Add food names
target_whole_foods_df["food_name"] = target_whole_foods_df["fdc_id"].map(whole_foods_id_map)
target_whole_foods_df

,fdc_id,protein,fat,carbohydrate,Energy (Atwater Specific Factors),food_name
0,329490,48.100000,39.8000,1.870000,576.000000,egg
1,331897,23.900000,5.9500,0.000000,156.000000,chicken_wings
2,746763,27.300000,11.4000,0.000000,219.000000,beef
3,746764,0.810000,0.4700,7.920000,37.000000,carrots
4,747448,0.640000,0.2200,7.630000,31.000000,strawberries
5,1105314,0.740000,0.2900,23.000000,88.000000,banana
6,1750339,0.187500,0.2125,14.781700,55.622745,apple
7,1750347,2.890625,0.3708,4.079375,24.873258,mushrooms


All amounts are per 100g.

## Export first 10 target food nutrition information

In [228]:
target_whole_foods_df.to_csv("target_ten_whole_food_nutrition_info.csv", index=False)

In [229]:
ten_whole_foods

['chicken_wings',
 'apple',
 'banana',
 'beef',
 'carrots',
 'egg',
 'strawberries',
 'blueberries',
 'mushrooms',
 'honey']

In [ ]:
foundation_food.head(-10)

,fdc_id,data_type,description,food_category_id,publication_date
651,321358,foundation_food,"hummus, commercial",16.0,2019-04-01
652,321359,foundation_food,"milk, reduced fat, fluid, 2% milkfat, with add...",1.0,2019-04-01
653,321360,foundation_food,"tomatoes, grape, raw",11.0,2019-04-01
798,321505,foundation_food,"salt, table, iodized",2.0,2019-04-01
904,321611,foundation_food,"beans, snap, green, canned, regular pack, drai...",11.0,2019-04-01
...,...,...,...,...,...
93407,2710800,survey_fndds_food,"cabbage, cooked, as ingredient",9999.0,2022-10-28
93408,2710801,survey_fndds_food,"cauliflower, cooked, as ingredient",9999.0,2022-10-28
93409,2710802,survey_fndds_food,"eggplant, cooked, as ingredient",9999.0,2022-10-28
93410,2710803,survey_fndds_food,"green beans, cooked, as ingredient",9999.0,2022-10-28


: 

# Mine

In [52]:
foundation_food["data_type"].unique()

<ArrowStringArray>
['foundation_food', 'survey_fndds_food']
Length: 2, dtype: str

In [53]:
foundation_food["description"].unique()

<ArrowStringArray>
[                                                        'hummus, commercial',
   'milk, reduced fat, fluid, 2% milkfat, with added vitamin a and vitamin d',
                                                       'tomatoes, grape, raw',
                                                       'salt, table, iodized',
                   'beans, snap, green, canned, regular pack, drained solids',
                                                              'broccoli, raw',
        'milk, lowfat, fluid, 1% milkfat, with added vitamin a and vitamin d',
 'milk, nonfat, fluid, with added vitamin a and vitamin d (fat free or skim)',
                           'milk, whole, 3.25% milkfat, with added vitamin d',
                                                'frankfurter, beef, unheated',
 ...
                                         'cauliflower, cooked, as ingredient',
                                            'eggplant, cooked, as ingredient',
                            

In [54]:
foundation_foods[50:100]

10162    turkey, ground, 93% lean, 7% fat, pan-broiled ...
11190    chicken, broilers or fryers, drumstick, meat o...
11253    chicken, broiler or fryers, breast, skinless, ...
11575     sauce, pasta, spaghetti/marinara, ready-to-serve
11690    ham, sliced, pre-packaged, deli meat (96%fat f...
11890    pears, raw, bartlett (includes foods for usda'...
12084     olives, green, manzanilla, stuffed with pimiento
12157    sausage, pork, chorizo, link or ground, cooked...
12301                 cookies, oatmeal, soft, with raisins
12574                   tomatoes, canned, red, ripe, diced
12667                                   fish, haddock, raw
12769                                   fish, pollock, raw
13487    fish, tuna, light, canned in water, drained so...
13540                                   sugars, granulated
13755             restaurant, chinese, sweet and sour pork
13829        restaurant, chinese, fried rice, without meat
13921                     restaurant, latino, tamale, po

In [55]:
for food in foundation_foods:
    if "chicken" in food.lower():
        print(food)

chicken, broilers or fryers, drumstick, meat only, cooked, braised
chicken, broiler or fryers, breast, skinless, boneless, meat only, cooked, braised
mock chicken legs, cooked
chicken, ns as to part and cooking method, ns as to skin eaten
chicken, ns as to part and cooking method, skin eaten
chicken, ns as to part and cooking method, skin not eaten
chicken, ns as to part, baked, broiled, or roasted, ns as to skin eaten
chicken, ns as to part, baked, broiled, or roasted, skin eaten
chicken, ns as to part, baked, broiled, or roasted, skin not eaten
chicken, ns as to part, rotisserie, ns as to skin eaten
chicken, ns as to part, rotisserie, skin eaten
chicken, ns as to part, rotisserie, skin not eaten
chicken, ns as to part, stewed, ns as to skin eaten
chicken, ns as to part, stewed, skin eaten
chicken, ns as to part, stewed, skin not eaten
chicken, ns as to part, grilled without sauce, ns as to skin eaten
chicken, ns as to part, grilled without sauce, skin eaten
chicken, ns as to part, gr

In [56]:
foundation_food[foundation_food["description"].str.contains("chicken", case=False)]

,fdc_id,data_type,description,food_category_id,publication_date
11190,331897,foundation_food,"chicken, broilers or fryers, drumstick, meat o...",5.0,2019-04-01
11253,331960,foundation_food,"chicken, broiler or fryers, breast, skinless, ...",5.0,2019-04-01
28471,1098388,survey_fndds_food,"mock chicken legs, cooked",NaN,2020-10-30
28501,1098418,survey_fndds_food,"chicken, ns as to part and cooking method, ns ...",NaN,2020-10-30
28502,1098419,survey_fndds_food,"chicken, ns as to part and cooking method, ski...",NaN,2020-10-30
...,...,...,...,...,...
33874,1103791,survey_fndds_food,"vegetable and chicken, baby food, ns as to str...",NaN,2020-10-30
33875,1103792,survey_fndds_food,"vegetable and chicken, baby food, strained",NaN,2020-10-30
33876,1103793,survey_fndds_food,"vegetable and chicken, baby food, junior",NaN,2020-10-30
33883,1103800,survey_fndds_food,"potato chicken pie, puerto rican style",NaN,2020-10-30


In [57]:
nutrient[nutrient["name"].str.contains("protein", case=False) | \
         nutrient["name"].str.contains("carbohydrate", case=False) | \
         nutrient["name"].str.contains("fat", case=False)]["name"].unique()

<ArrowStringArray>
[                                                             'Protein',
                                                    'Total lipid (fat)',
                                          'Carbohydrate, by difference',
                                                      'Solids, non-fat',
                                           'Carbohydrate, by summation',
                                                     'Adjusted Protein',
                                                  'Carbohydrate, other',
                                                     'Total fat (NLEA)',
                                             'Fatty acids, total trans',
                                         'Fatty acids, total saturated',
 'Fatty acids, other than 607-615, 617-621, 624-632, 652-654, 686-689)',
                                   'Fatty acids, total monounsaturated',
                                   'Fatty acids, total polyunsaturated',
                                

Getting IDs of Carbohydrate, Fat and Protein

* Carbohydrate, by difference = total carbohydrate

In [58]:
target_nutrients = nutrient[nutrient["name"].isin(["Protein", "Carbohydrate, by difference", "Total lipid (fat)"])]
target_nutrients

,id,name,unit_name,nutrient_nbr,rank
2,1003,Protein,G,203.0,600.0
3,1004,Total lipid (fat),G,204.0,800.0
4,1005,"Carbohydrate, by difference",G,205.0,1110.0


In [59]:
food_nutrient[(food_nutrient["fdc_id"] ==  chicken_wing_id) & (food_nutrient["nutrient_id"].isin(list(target_nutrient_dict.keys())))]

,id,fdc_id,nutrient_id,amount,data_points,derivation_id,min,max,median,footnote,min_year_acqured,sf.footnote,min_year_acquired,nutrient_name
41686,2259098,331897,1004,5.95,6.0,1.0,5.54,6.33,5.93,NaN,2010.0,NaN,NaN,total lipid (fat)
41718,2259079,331897,1003,23.90,NaN,49.0,23.00,24.60,24.10,NaN,NaN,NaN,NaN,protein
41729,2259099,331897,1005,0.00,NaN,49.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"carbohydrate, by difference"


Getting Target Nutrient(protein, total lipid(fat), carbohydrate, by difference) of `chicken_wing_id`

In [74]:
food_nutrient[(food_nutrient["fdc_id"] == chicken_wing_id) & food_nutrient["nutrient_id"].isin(list(target_nutrient_dict.keys()))]

,id,fdc_id,nutrient_id,amount,data_points,derivation_id,min,max,median,footnote,min_year_acqured,sf.footnote,min_year_acquired,nutrient_name
41686,2259098,331897,1004,5.95,6.0,1.0,5.54,6.33,5.93,NaN,2010.0,NaN,NaN,total lipid (fat)
41718,2259079,331897,1003,23.90,NaN,49.0,23.00,24.60,24.10,NaN,NaN,NaN,NaN,protein
41729,2259099,331897,1005,0.00,NaN,49.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"carbohydrate, by difference"


In [95]:
food[(food["description"].str.contains("blueberry", case=False))]["description"]

31153    pie, berry, not blackberry, blueberry, boysenb...
31154    pie, berry, not blackberry, blueberry, boysenb...
31155    pie, berry, not blackberry, blueberry, boysenb...
31156                            pie, blueberry, two crust
31157              pie, blueberry, individual size or tart
31239                                     crisp, blueberry
31848           cereal (malt-o-meal blueberry muffin tops)
31888               cereal (kellogg's special k blueberry)
32788                                blueberry pie filling
32833                                      blueberry juice
32939        blueberry yogurt dessert, baby food, strained
34033                                      blueberry syrup
Name: description, dtype: str

In [100]:
# food_nutrient[(food_nutrient["fdc_id"] == chicken_curry_id) & food_nutrient["nutrient_id"].isin(list(target_nutrient_dict.keys()))]
chicken_curry_id = int(foundation_food.loc[foundation_food["description"].str.contains(pattern, case=False)].iloc[0]["fdc_id"])
chicken_curry_id

323121

**Code Snip!!!**

Setting Food Id \
`chicken_wing_id = int(foundation_food.loc[foundation_food["description"].str.contains("Chicken", case=False)].iloc[0]["fdc_id"])`

Getting Nutrition Data from ID \
`food_nutrient[(food_nutrient["fdc_id"] == chicken_wing_id) & food_nutrient["nutrient_id"].isin(list(target_nutrient_dict.keys()))]`

In [82]:
chicken_curry_id = int(foundation_food.loc[foundation_food["description"].str.contains("chicken curry", case=False)].iloc[0]["fdc_id"])
food_nutrient[(food_nutrient["fdc_id"] == chicken_curry_id) & food_nutrient["nutrient_id"].isin(list(target_nutrient_dict.keys()))]

,id,fdc_id,nutrient_id,amount,data_points,derivation_id,min,max,median,footnote,min_year_acqured,sf.footnote,min_year_acquired,nutrient_name
218537,12988060,1099246,1005,6.47,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"carbohydrate, by difference"
218540,12988058,1099246,1003,6.47,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,protein
218555,12988059,1099246,1004,6.45,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,total lipid (fat)


In [172]:
# Check if 2048 nutrient entries exist for our target foods
fdc_ids_with_2048 = food_nutrient[
    (food_nutrient["fdc_id"].isin(list(whole_foods_id_map.keys()))) &
    (food_nutrient["nutrient_id"] == 2048)
]
print(f"Foods with Energy (Atwater Specific Factors) recorded: {len(fdc_ids_with_2048)}")
fdc_ids_with_2048[["fdc_id", "amount"]].merge(
    pd.DataFrame(whole_foods_id_map.items(), columns=["fdc_id", "food_name"]),
    on="fdc_id",
    how="right"
)

Foods with Energy (Atwater Specific Factors) recorded: 2


,fdc_id,amount,food_name
0,1750339,55.622745,apple
1,1105314,NaN,banana
2,1102702,NaN,blueberries
3,746763,NaN,beef
4,746764,NaN,carrots
5,331897,NaN,chicken_wings
6,329490,NaN,egg
7,1103956,NaN,honey
8,1750347,24.873258,mushrooms
9,747448,NaN,strawberries
